In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
df=pd.read_csv('/kaggle/input/datasets/organizations/zalando-research/fashionmnist/fashion-mnist_train.csv')

In [ ]:
import torch
import matplotlib.pyplot as plt


In [ ]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
X=df.iloc[:,1:].values
y=df.iloc[:,0].values

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(X,y,test_size=0.2)

In [ ]:
x_train_scaled=x_train/255
x_test_scaled=x_test/255

In [ ]:
x_train_tensor=torch.from_numpy(x_train_scaled).float()
x_test_tensor=torch.from_numpy(x_test_scaled).float()
y_train_tensor=torch.from_numpy(y_train).long()
y_test_tensor=torch.from_numpy(y_test).long()

In [ ]:
from torch.utils.data import Dataset,DataLoader

In [ ]:
class Customdataset(Dataset):
  def __init__(self,x,y):
    self.x=x
    self.y=y
  def __len__(self):
    return len(self.x)
  def __getitem__(self, index):
    return self.x[index],self.y[index]

In [ ]:
train_dataset=Customdataset(x_train_tensor,y_train_tensor)
test_dataset=Customdataset(x_test_tensor,y_test_tensor)

In [ ]:
import torch.nn as nn
from sklearn.metrics import accuracy_score

In [ ]:
class NewNeuralNetwork(nn.Module):
  def __init__(self,input_dim,output_dim,dropout_rate,neurons_per_layers):
    super().__init__()
    layers=[]
    for neurons_per_layer,layer_dropout in zip(neurons_per_layers, dropout_rate):
      layers.append(nn.Linear(input_dim, neurons_per_layer))
      layers.append(nn.BatchNorm1d(neurons_per_layer))
      layers.append(nn.ReLU())
      layers.append(nn.Dropout(layer_dropout))
      input_dim = neurons_per_layer
    layers.append(nn.Linear(input_dim,output_dim))
    self.model=nn.Sequential(*layers) 
  def forward(self,X):
    return self.model(X)


In [ ]:
def optuna_tuning(trial):
  num_hidden_layers=trial.suggest_int('num_hidden_layers',1,5)
  neurons_per_layers=[]
  dropout_rates=[]
  for i in range(num_hidden_layers):
      layer_size = trial.suggest_int(f"layer_{i}_size", 16,256, step=16)
      neurons_per_layers.append(layer_size)
      layer_dropout = trial.suggest_float(f"dropout_layer_{i}", 0.1, 0.5, step=0.1)
      dropout_rates.append(layer_dropout)
  learning_rate=trial.suggest_float('learning_rate',1e-5,1e-1,log=True)
  dropout_rate=trial.suggest_float('dropout_rate',0.1,0.5,step=0.1)
  epochs=trial.suggest_int('epochs',10,100,step=10)
  batch_size=trial.suggest_int('batch_size',16,128,step=16)
  optimizer_name=trial.suggest_categorical('optimizer',['adam','sgd','RMSProp'])
  weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)

  test_dataloader=DataLoader(test_dataset,batch_size=batch_size,shuffle=True)
  train_dataloader=DataLoader(train_dataset,batch_size=batch_size,shuffle=True)

  input_dim=x_train_tensor.shape[1]
  output_dim=10

  model=NewNeuralNetwork(input_dim,output_dim,dropout_rates,neurons_per_layers)
  model.to(device)
  loss_function=nn.CrossEntropyLoss()
  if optimizer_name=='adam':
    optimizer=torch.optim.Adam(model.parameters(),lr=learning_rate,weight_decay=weight_decay)
  elif optimizer_name=='sgd':
    optimizer=torch.optim.SGD(model.parameters(),lr=learning_rate,weight_decay=weight_decay)
  elif optimizer_name=='RMSProp':
    optimizer=torch.optim.RMSprop(model.parameters(),lr=learning_rate,weight_decay=weight_decay)
  for i in range(epochs):
    for x_train_batch,y_train_batch in train_dataloader:
      x_train_batch,y_train_batch=x_train_batch.to(device),y_train_batch.to(device)
      y_pred=model(x_train_batch)
      loss=loss_function(y_pred,y_train_batch.long())
      loss.backward()
      optimizer.step()
      optimizer.zero_grad()
  model.eval()
  accuracy_list=[]
  with torch.no_grad():
    for x_test_batch,y_test_batch in test_dataloader:
      x_test_batch,y_test_batch=x_test_batch.to(device),y_test_batch.to(device)
      y_pred=model(x_test_batch)
      y_pred=torch.argmax(y_pred,dim=1)
      accuracy=accuracy_score(y_pred.cpu().numpy(),y_test_batch.cpu().numpy())
      accuracy_list.append(accuracy)
    result=sum(accuracy_list)/len(accuracy_list)
  return result

In [ ]:
!pip install optuna

In [ ]:
import optuna
study = optuna.create_study(direction='maximize')

In [ ]:
study.optimize(optuna_tuning,n_trials=20)

In [ ]:
study.best_value

In [ ]:
study.best_params

In [ ]:
x_full_scaled=X/255

In [ ]:
x_full_tensor=torch.from_numpy(x_full_scaled).float()
y_full_tensor=torch.from_numpy(y).long()

In [ ]:
full_train_dataset=Customdataset(x_full_tensor,y_full_tensor)

In [ ]:
df_sub=pd.read_csv('/kaggle/input/datasets/organizations/zalando-research/fashionmnist/fashion-mnist_test.csv')
x_sub=df_sub.iloc[:,1:].values
y_sub=df_sub.iloc[:,0].values
x_sub_scaled=x_sub/255
x_sub_tensor=torch.from_numpy(x_sub_scaled).float()
y_sub_tensor=torch.from_numpy(y_sub).long()
full_sub_dataset=Customdataset(x_sub_tensor,y_sub_tensor)
full_sub_dataloader=DataLoader(full_sub_dataset,batch_size=study.best_params["batch_size"],shuffle=False)

In [ ]:
best_params = study.best_params
num_layers = best_params["num_hidden_layers"]
full_train_dataloader=DataLoader(full_train_dataset,batch_size=best_params["batch_size"],shuffle=True)
best_hidden_dims = [best_params[f"layer_{i}_size"] for i in range(num_layers)]
best_dropout_rates = [best_params[f"dropout_layer_{i}"] for i in range(num_layers)]
final_model = NewNeuralNetwork(input_dim=x_train_tensor.shape[1],output_dim=10,dropout_rate=best_dropout_rates,neurons_per_layers=best_hidden_dims)
final_model.to(device)
best_epochs=best_params["epochs"]
best_optimizer=best_params["optimizer"]
best_lr=best_params["learning_rate"]
best_weight_decay=best_params["weight_decay"]
loss_function=nn.CrossEntropyLoss()

if best_optimizer=='adam':
    optimizer=torch.optim.Adam(final_model.parameters(),lr=best_lr,weight_decay=best_weight_decay)
elif best_optimizer=='sgd':
    optimizer=torch.optim.SGD(final_model.parameters(),lr=best_lr,weight_decay=best_weight_decay)
elif best_optimizer=='RMSProp':
    optimizer=torch.optim.RMSprop(model.parameters(),lr=best_lr,weight_decay=best_weight_decay)
for i in range(best_epochs):
    for x_train_batch,y_train_batch in full_train_dataloader:
      x_train_batch,y_train_batch=x_train_batch.to(device),y_train_batch.to(device)
      y_pred=final_model(x_train_batch)
      loss=loss_function(y_pred,y_train_batch.long())
      loss.backward()
      optimizer.step()
      optimizer.zero_grad()


In [ ]:
final_model.eval()
train_accuracy_list=[]
with torch.no_grad():
    for x_train_batch,y_train_batch in full_train_dataloader:
      x_train_batch,y_train_batch=x_train_batch.to(device),y_train_batch.to(device)
      y_pred=final_model(x_train_batch)
      y_pred=torch.argmax(y_pred,dim=1)
      accuracy=accuracy_score(y_pred.cpu().numpy(),y_train_batch.cpu().numpy())
      train_accuracy_list.append(accuracy)
    result=sum(train_accuracy_list)/len(train_accuracy_list)
print("Training Data Accuracy: ",result)

In [ ]:
final_model.eval()
test_accuracy_list=[]
with torch.no_grad():
    for x_sub_batch,y_sub_batch in full_sub_dataloader:
      x_sub_batch,y_sub_batch=x_sub_batch.to(device),y_sub_batch.to(device)
      y_pred=final_model(x_sub_batch)
      y_pred=torch.argmax(y_pred,dim=1)
      accuracy=accuracy_score(y_pred.cpu().numpy(),y_sub_batch.cpu().numpy())
      test_accuracy_list.append(accuracy)
    result=sum(test_accuracy_list)/len(test_accuracy_list)
print("Submission Data Accuracy: ",result)